## ML Model Training and Experiments

- Converting data for ML model Consumable
- Adding Weight & Biases for Experiment Tracking 
- Training Different Model 
- Evaluations

#### Data Conversion

In [1]:
# imports
import pandas as pd

In [2]:
df = pd.read_csv("../dataset/ml_raw_data.csv")
df.head()

,age,sex,resting_bp_systolic,resting_bp_diastolic,hdl,ldl,triglycerides,hba1c,bmi,resting_heart_rate,...,family_history,smoker_status,alcohol_units_per_week,exercise_minutes_per_week,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease
0,44,Male,117,74,57,106,119,5.2,26.5,74,...,False,Never,2.9,86,5.4,19.8,True,7731,62.9,0
1,57,Male,139,94,69,110,35,5.9,20.8,63,...,False,Never,3.0,132,4.3,45.8,True,2629,74.6,1
2,29,Male,128,78,52,108,157,5.5,24.5,81,...,False,Current,3.5,128,5.1,17.7,True,9290,65.7,0
3,72,Male,132,86,59,104,143,4.8,27.3,93,...,False,Never,2.7,18,6.8,63.6,True,7373,48.5,1
4,62,Female,116,75,65,75,104,6.2,26.0,86,...,True,Former,3.3,24,8.2,58.7,False,6331,47.3,1


In [3]:
df.columns.groupby(df.dtypes)

{int64: ['age', 'resting_bp_systolic', 'resting_bp_diastolic', 'hdl', 'ldl', 'triglycerides', 'resting_heart_rate', 'max_heart_rate_achieved', 'exercise_minutes_per_week', 'daily_steps', 'has_heart_disease'], str: ['sex', 'chest_pain_type', 'smoker_status'], float64: ['hba1c', 'bmi', 'st_depression', 'alcohol_units_per_week', 'sleep_hours', 'stress_score', 'diet_quality_score'], bool: ['exercise_induced_angina', 'family_history', 'wearable_owner']}

In [4]:
NUMERICAL_FEATURE = ['age', 'resting_bp_systolic', 'resting_bp_diastolic', 'hdl', 'ldl', 'triglycerides', 'resting_heart_rate', 'max_heart_rate_achieved', 'exercise_minutes_per_week', 'daily_steps','hba1c', 'bmi', 'st_depression', 'alcohol_units_per_week', 'sleep_hours', 'stress_score', 'diet_quality_score'] 
CATEGORICAL_FEATURE = ['sex', 'chest_pain_type', 'smoker_status', 'exercise_induced_angina', 'family_history', 'wearable_owner'] 
TARGET_FEATURE = 'has_heart_disease'


In [5]:
# Normalizing the numerical feature 
df[NUMERICAL_FEATURE].head()

,age,resting_bp_systolic,resting_bp_diastolic,hdl,ldl,triglycerides,resting_heart_rate,max_heart_rate_achieved,exercise_minutes_per_week,daily_steps,hba1c,bmi,st_depression,alcohol_units_per_week,sleep_hours,stress_score,diet_quality_score
0,44,117,74,57,106,119,74,165,86,7731,5.2,26.5,0.3,2.9,5.4,19.8,62.9
1,57,139,94,69,110,35,63,150,132,2629,5.9,20.8,2.1,3.0,4.3,45.8,74.6
2,29,128,78,52,108,157,81,210,128,9290,5.5,24.5,0.8,3.5,5.1,17.7,65.7
3,72,132,86,59,104,143,93,137,18,7373,4.8,27.3,2.1,2.7,6.8,63.6,48.5
4,62,116,75,65,75,104,86,181,24,6331,6.2,26.0,1.3,3.3,8.2,58.7,47.3


In [6]:
X = df[NUMERICAL_FEATURE + CATEGORICAL_FEATURE]
y = df[TARGET_FEATURE]

In [7]:
y.head()

0    0
1    1
2    0
3    1
4    1
Name: has_heart_disease, dtype: int64

In [8]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
        [
            ('num-scaler',StandardScaler(), NUMERICAL_FEATURE),
            ('cat-encoder',OneHotEncoder(),CATEGORICAL_FEATURE)
        ]
)



In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test , y_train, y_test = train_test_split(X,y,train_size=0.8)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((7200, 23), (1800, 23), (7200,), (1800,))

## Experimenting with Different ML Algorithms

- Random Forest Classifier
- Linear Classifier
- SVM Classifier


In [10]:
# common imports
import wandb
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_auc_score, recall_score
import joblib


In [14]:
import joblib
import wandb
import wandb.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

# 1. Initialize W&B Run
run_rf = wandb.init(
    project="mtech-demo-experiments",
    name="random-forest-run-v5",
    config={
        "algorithm": "RandomForest",
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42,
    },
)

# 2. Build and Fit Pipeline
rforest_classifier = Pipeline(
    steps=[
        ("data-preprocessor", preprocessor),
        (
            "rforest-classifier",
            RandomForestClassifier(
                n_estimators=100, max_depth=10, random_state=42
            ),
        ),
    ]
)

rforestM = rforest_classifier.fit(X_train, y_train)

# 3. Predict Classes & Probabilities
y_pred = rforestM.predict(X_test)
y_probas = rforestM.predict_proba(X_test)

# Define class labels for W&B plots (adjust names to match your dataset)
class_names = [str(cls) for cls in rforestM.classes_]

# 4. Scalar Metrics Calculation
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

# Handle binary vs multiclass ROC AUC scoring
if len(class_names) == 2:
    auc = roc_auc_score(y_test, y_probas[:, 1])
else:
    auc = roc_auc_score(y_test, y_probas, multi_class="ovr")

print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | ROC AUC: {auc:.4f}")

# 5. Log Scalar Metrics
wandb.log({
    "accuracy": acc,
    "precision": prec,
    "recall": recall,
    "f1_score": f1,
    "roc_auc_score": auc,
})

# ==============================================================================
# INDIVIDUAL PLOTS (wandb.sklearn & wandb.plot)
# ==============================================================================

# Individual 1: Confusion Matrix
wandb.sklearn.plot_confusion_matrix(y_test, y_pred, labels=class_names)

# Individual 2: ROC Curve
wandb.sklearn.plot_roc(y_test, y_probas, labels=class_names)

# Individual 3: Precision-Recall Curve
wandb.sklearn.plot_precision_recall(y_test, y_probas, labels=class_names)

# Individual 4: Feature Importances
# Extracts the trained classifier from the pipeline step
rf_model = rforestM.named_steps["rforest-classifier"]
if hasattr(preprocessor, "get_feature_names_out"):
    feature_names = preprocessor.get_feature_names_out()
else:
    feature_names = [f"feature_{i}" for i in range(X_train.shape[1])]

wandb.sklearn.plot_feature_importances(rf_model, feature_names=feature_names)

# Individual 5: Learning Curve
wandb.sklearn.plot_learning_curve(rf_model, X_train, y_train)

# ==============================================================================
# 6. Save Model Artifact
# ==============================================================================
joblib.dump(rforest_classifier, "../models/rforest_classifier.joblib")

artifact = wandb.Artifact(
    name="RandomForestClassifier",
    type="model",
    description="Random Forest pipeline model with preprocessor",
)
artifact.add_file("../models/rforest_classifier.joblib")

run_rf.log_artifact(artifact)

wandb.finish()

Accuracy: 0.8817 | Precision: 0.8813 | Recall: 0.8817 | F1: 0.8785 | ROC AUC: 0.9303


wandb: ERROR X contains values that are not numbers. Please vectorize, label encode or one hot encode X and call the plotting function again.
wandb: WARNING Artifact "RandomForestClassifier" already exists with the same content. No new version will be created.


accuracy,▁
f1_score,▁
precision,▁
recall,▁
roc_auc_score,▁
accuracy,0.88167
f1_score,0.87854
precision,0.88132
recall,0.88167
roc_auc_score,0.93032
